In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
from typing import TypedDict

load_dotenv()

True

In [2]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ.get("GROK_API_KEY")
)

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    

In [4]:
def create_outline(state: BlogState) -> BlogState:

    title = state["title"]

    prompt = f'Generate an outline for a blog on the topic "{title}"'

    outline = model.invoke(prompt)

    state["outline"] = outline.content

    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state["title"]

    outline = state["outline"]

    prompt = f"""
    Write a detailed blog on the title "{title}"
    using the following outline:

    {outline}
    """

    content = model.invoke(prompt)

    state["content"] = content.content

    return state

In [6]:
from langgraph.graph  import StateGraph,START,END

graph = StateGraph(BlogState)
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)
workflow=graph.compile()

In [7]:
initial_state = {
    "title": "Rise of AI in India"
}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '# Blog Outline  \n**Title:** *The Rise of AI in India: From Dream to Reality*  \n\n---\n\n## 1. Hook & Introduction (≈200‑250 words)  \n- **Opening anecdote** – a quick story of an AI‑driven startup solving a local problem (e.g., rural diagnostics, traffic optimization).  \n- **Why it matters** – India’s demographic dividend + digital push.  \n- **Thesis statement** – India is rapidly moving from “AI‑hope” to “AI‑impact,” reshaping its economy, society, and global standing.  \n\n### Key Questions to Answer  \n- What has fueled AI’s growth in India?  \n- Which sectors are leading the charge?  \n- What challenges still loom?  \n\n---\n\n## 2. Historical Context: From “AI‑Hype” to “AI‑Reality” (≈300 words)  \n- **Early 2000s** – academic research, AI clubs, and the first AI conferences.  \n- **2010s** – the rise of data‑driven startups, increased venture capital, and the “AI for All” mantra.  \n- **2020‑Present** – mainstream adoption, AI in go